In [74]:
import numpy as np

In [75]:
X = [
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
]

Creating the shape, transpose and the multiplication function

In [76]:
# Level 0 and 1 

def matrix_shape(A):
    count = len(A[0])
    return (len(A), count)


def transpose(A):
    return [[A[i][j] for i in range(len(A))] for j in range(len(A[0]))]

def matrix_multiply(A, B):
    if matrix_shape(A)[1] != matrix_shape(B)[0]:
        return 'Matrix Multiplication is not possible.'
    else:
        B_t = transpose(B)
        result = []
        for row_a in A:
            new_row = []
            for row_b_t in B_t:
                element = sum(a * b for a, b in zip(row_a, row_b_t))
                new_row.append(element)
        
            result.append(new_row)

        return result


# Define Regions

def extract_region(X, row_start:int, col_start:int, size:int):
    element = [[X[row_start+i][col_start+j] for j in range(size)] for i in range(size)]
    return element



# Create single patch

def extract_patch(X, patch_row: int, patch_col: int, patch_size: int):
    start_row = patch_row * patch_size
    end_row = start_row + patch_size

    start_col = patch_col * patch_size
    end_col = start_col + patch_size

    return [row[start_col:end_col] for row in X[start_row:end_row]]

print('Done ✅')

Done ✅


In [77]:
# Level 2


# Number of Patches
def number_of_patches(X, P:int)->int:
    H = matrix_shape(X)[0]
    W = matrix_shape(X)[1]

    Nh = H // P
    Nw = W // P

    return Nh * Nw

# Extracting multiple patches

def extract_patches(X, P: int):
    H = len(X)
    W = len(X[0])
    
    Nh = H // P
    Nw = W // P

    N = Nh * Nw
    
    patches = []
    for p in range(N):
        patch = extract_patch(X, p // Nw, p % Nw, P)
        patches.append(patch)
        
    return patches

print('Done ✅')

Done ✅


In [78]:
# Level 3 flattened patch/patches for image pixel matrix

def flatten_patch(patch):
    x = [element for row in patch for element in row]
    return x

def flatten_patches(patches):
    '''
    This always returns N x P^2 flattened patches.
    '''
    N = len(patches)
    flattened_patches = []
    for p in range(N):
        flattened_patches.append(flatten_patch(patches[p]))
    return flattened_patches

print('Done ✅')

Done ✅


In [79]:
# Level 4 porjection patch matrix creation

def create_matrix(r, c, value):
    return [[value for j in range(c)] for i in range(r)]

import random

def random_matrix(r, c, seed=None):
    random.seed(seed)
    return [[random.random() for j in range(c)] for i in range(r)]


X_patches = flatten_patches(extract_patches(X, 2))
N = matrix_shape(X_patches)[0]
P_squared = matrix_shape(X_patches)[1]
D = 4
W_E = random_matrix(P_squared, D, 42)

def project_patches(X_patches, W_E):
    return matrix_multiply(X_patches, W_E)


T = project_patches(X_patches, W_E)
b = [0 for _ in range(D)]

def add_bias(T, b):
    result = []
    for row in T:
        new_row = [row[j] + b[j] for j in range(len(row))]
        result.append(new_row)

    return result

C = add_bias(T, b)
print(C[:2])
print('Done ✅')

[[4.381194143315443, 2.720421731378732, 7.051884954473966, 6.193513727543767], [8.029905747297503, 4.581111956919326, 11.123347551788148, 8.914406406513404]]
Done ✅


In [80]:
# Level 5 Embedding matrix

def create_position_matrix(N, D):
    '''
    Position Embedding Matrix
    '''
    matrix = []
    for i in range(N):
        row = []
        for j in range(D):
            row.append(i + j)
        matrix.append(row)
    return matrix

def add_position_embeddings(T, E_pos):
    '''
    Adding Embedding Matrix
    '''
    result = []
    for i in range(len(T)):
        row = []
        for j in range(len(T[0])):
            row.append(T[i][j] + E_pos[i][j])
        result.append(row)
    return result

E_pos = create_position_matrix(N, D)

Z = add_position_embeddings(C, E_pos)

print(Z[:2])
print('Done ✅')

[[4.381194143315443, 3.720421731378732, 9.051884954473966, 9.193513727543767], [9.029905747297503, 6.581111956919326, 14.123347551788148, 12.914406406513404]]
Done ✅


In [81]:
# Level 6

def tokenize_image(X, P, D):
    X_patches = extract_patches(X, P = P)
    X_flatten = flatten_patches(X_patches)

    P_squared = P ** 2

    W_E = random_matrix(P_squared, D, seed = 42)

    T = project_patches(X_flatten, W_E)
    
    b = [0 for _ in range(D)]

    N = matrix_shape(X_patches)[0]

    C = add_bias(T, b)

    E_pos = create_position_matrix(N, D)

    Z = add_position_embeddings(C, E_pos)

    return Z

print(tokenize_image(X, P = 4, D = 16))

[[53.9938923123651, 82.78759848616433, 66.40425769802242, 97.5725819124271, 63.61830969261926, 78.43645230103253, 85.99799140578683, 77.65638006931886, 78.27202720936403, 77.99168685246975, 73.20677218801313, 59.92572725442129, 79.95539978324679, 78.52037373130634, 83.11631445771972, 85.32945428403947]]
